# {{PROJECT_NAME}} — Explainability

**Owner(s):**  
**Goal:** Understand what drives the champion model's predictions using SHAP.

Outputs from this notebook feed into `src/explain.py` and the app's Case Review page.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib

shap.initjs()

## 1. Load champion model and validation data

In [ ]:
model   = joblib.load('../models/champion.pkl')
X_val   = pd.read_parquet('../data/processed/X_val.parquet')
y_val   = pd.read_parquet('../data/processed/y_val.parquet').squeeze()

print(f'Val: {X_val.shape}')

## 2. Build SHAP explainer

In [ ]:
# Use a background sample of 200-500 rows for speed
background = X_val.sample(min(300, len(X_val)), random_state=42)

explainer = shap.TreeExplainer(
    model,
    data=background,
    feature_perturbation='interventional',
)
shap_values = explainer.shap_values(X_val)

## 3. Global feature importance

In [ ]:
shap.summary_plot(shap_values, X_val, plot_type='bar', max_display=20)

## 4. SHAP beeswarm — direction and magnitude

In [ ]:
shap.summary_plot(shap_values, X_val, max_display=20)

## 5. Per-prediction waterfall — single flagged record

TODO: replace `idx` with the index of an interesting flagged record.

In [ ]:
idx = 0  # TODO: pick a representative flagged record

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_val.iloc[idx],
        feature_names=X_val.columns.tolist(),
    )
)

## 6. Key findings — update MODEL_CARD.md

Document the top drivers and any surprising results:

- **Top 3 features:** TODO
- **Any unexpected drivers:** TODO
- **Features with low SHAP despite high correlation:** TODO
- **Anything to flag for {{PERSONA}}:** TODO